In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_absolute_error

In [ ]:
folder_with_data = '../Data/Final-Data'

river_flow = pd.read_csv(folder_with_data + '/river-flow.csv')
river_flow['dateTime'] = pd.to_datetime(river_flow['dateTime'], errors='raise').dt.date
river_flow = river_flow.groupby('dateTime').mean().reset_index()

climate_variables = pd.read_csv(folder_with_data + '/atmospheric-variables.csv')
climate_variables['dateTime'] = pd.to_datetime(
    climate_variables['dateTime'],
    errors='raise'
)

numeric_cols = climate_variables.select_dtypes(include=np.number).columns
climate_variables[numeric_cols] = climate_variables[numeric_cols].interpolate(
    method='linear',
    limit_direction='both'
)

climate_variables['dateTime'] = climate_variables['dateTime'].dt.date

river_flow = river_flow.drop(columns='Unnamed: 0')

weather_data = pd.merge(
    climate_variables,
    river_flow,
    on='dateTime',
    how='inner'
)

weather_data = weather_data.iloc[:, [0,3,1,4,5,6,7,8,9,10,11,12,13]]

time_steps = 30
horizon = 7

numerical_columns = weather_data.select_dtypes(include=np.number).columns

In [3]:
actuals = np.load('actuals-un.npy')

In [4]:
def ensemble_predict(predictions): 
    predictions = np.array(predictions)
    avg_predictions = np.mean(predictions, axis=0)  
    return avg_predictions

def confidence_bands(predictions):
    predictions = np.array(predictions)
    mean = np.mean(predictions, axis=0)

    std = np.std(predictions, axis=0, ddof=1)

    n = predictions.shape[0]

    z = 1.96  # 95% confidence

    margin = z * std / np.sqrt(n)

    lower = mean - margin
    upper = mean + margin

    return lower, upper

def rob_prediction_intervals(predictions, actuals, confidence=0.95):
    """
    Calculates heteroscedastic prediction intervals using a global 
    log-space transform. This naturally pinches intervals at base flows
    and widens them at peaks.
    """
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    actuals = np.expand_dims(actuals, axis=-1)
    n_runs = predictions.shape[0]
        
    log_predictions = np.log(predictions + 1.0)
    log_actuals = np.log(actuals + 1.0)
    
    log_ensemble_mean = np.mean(log_predictions, axis=0)
    log_ensemble_var = np.var(log_predictions, axis=0, ddof=1)
    
    log_residuals = log_actuals - log_ensemble_mean
    
    print(f"Log Predictions shape: {log_predictions.shape}")
    print(f"Log Residuals shape:   {log_residuals.shape}")
    print(f"Log Actuals Max/Min:   {np.max(log_actuals):.3f} / {np.min(log_actuals):.3f}")
    print(f"Log Predictions Max/Min:   {np.max(log_predictions):.3f} / {np.min(log_predictions):.3f}")
    
    log_residual_var = np.var(log_residuals, axis=0, ddof=1, keepdims=True)
    
    log_total_std = np.sqrt(log_ensemble_var * (1 + 1/n_runs) + log_residual_var)
    
    z_map = {0.90: 1.645, 0.95: 1.96, 0.99: 2.576}
    z = z_map.get(confidence, 1.96)
    
    log_lower = log_ensemble_mean - (z * log_total_std)
    log_upper = log_ensemble_mean + (z * log_total_std)
    
    lower = np.exp(log_lower) - 1.0
    upper = np.exp(log_upper) - 1.0
    
    lower = np.clip(lower, a_min=0, a_max=None)
    
    return lower, upper

In [ ]:
predictions_per_model = np.load("predictions-per_model-un.npy")
prupper_per_model = np.load('prl-upper-predictions-per-model-un.npy')
pr_lower_per_model = np.load('prl-lower-predictions-per-model-un.npy')
upper_per_model = np.load('upper-predictions-per-model-un.npy')
lower_per_model  = np.load('lower-predictions-per-model-un.npy')

In [6]:
def nash_sutcliffe_efficiency(observed, predicted):
    observed = np.array(observed)
    predicted = np.array(predicted)

    numerator = np.sum((observed - predicted) ** 2)
    denominator = np.sum((observed - np.mean(observed)) ** 2)

    return 1 - (numerator / denominator)

def calculate_mase(actual, predicted, m, eps=1e-8):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    naive_error = np.abs(actual[m:] - actual[:-m])
    forecast_error = np.abs(actual[m:] - predicted[m:])

    denom = np.mean(naive_error)

    if denom < eps:
        return np.nan

    return np.mean(forecast_error) / denom

In [7]:
n_stations = 6
error_metrics = []

model_registry = {
    "LSTM-EncDec-DFF": {
        "preds": predictions_per_model[0],
        "lower": lower_per_model[0],
        "upper": upper_per_model[0]
    }
    ,
    "LSTM-EncDec-DFF-2": {
        "preds": predictions_per_model[1],
        "lower": lower_per_model[1],
        "upper": upper_per_model[1]
    }
}

def format_metric_with_bands(base_val, lower_val, upper_val):
    """Calculates the margin and formats as 'Base ± Margin'"""
    margin_lower = abs(base_val - lower_val)
    margin_upper = abs(upper_val - base_val)
    avg_margin = (margin_lower + margin_upper) / 2
    
    return f"{round(base_val, 2)} ± {round(avg_margin, 2)}"

for model_name, arrays in model_registry.items():
    preds_flat = np.array(arrays["preds"]).transpose(1,2,0).reshape(-1, n_stations).flatten()
    lower_flat = np.array(arrays["lower"]).transpose(1,2,0).reshape(-1, n_stations).flatten()
    upper_flat = np.array(arrays["upper"]).transpose(1,2,0).reshape(-1, n_stations).flatten()
    actuals_flat = actuals.flatten()

    df_actual = pd.DataFrame({'Flow': actuals_flat})
    threshold_high = df_actual['Flow'].quantile(0.95)  

    extreme_highs = df_actual[df_actual['Flow'] >= threshold_high]

    threshold_high2 = df_actual['Flow'].quantile(0.95)  
    threshold_low2 = df_actual['Flow'].quantile(0.05) 

    results = {}
    variants = {'base': preds_flat, 'lower': lower_flat, 'upper': upper_flat}  

    for v_name, v_data in variants.items():

        df_variant = pd.DataFrame({'Flow': v_data})
        extreme_highs_variant = df_variant[df_actual['Flow'] >= threshold_high]
        results[v_name] = {
            'mae': mean_absolute_error(df_actual['Flow'], df_variant['Flow']),
            'nse': nash_sutcliffe_efficiency(df_actual['Flow'], df_variant['Flow']),
            'mase': calculate_mase(df_actual['Flow'], df_variant['Flow'], 20),
            'mae_high': mean_absolute_error(extreme_highs['Flow'], extreme_highs_variant['Flow'])
        }
        
    model_metrics_df = pd.DataFrame({
        'Model': [model_name],
        'MAE': [format_metric_with_bands(results['base']['mae'], results['lower']['mae'], results['upper']['mae'])],
        'NSE': [format_metric_with_bands(results['base']['nse'], results['lower']['nse'], results['upper']['nse'])],
        'MASE': [format_metric_with_bands(results['base']['mase'], results['lower']['mase'], results['upper']['mase'])],
        'MAE_HIGH': [format_metric_with_bands(results['base']['mae_high'], results['lower']['mae_high'], results['upper']['mae_high'])]
    })

    error_metrics.append(model_metrics_df)
  
error_metrics = pd.concat(error_metrics, axis=0).reset_index(drop=True)
print(error_metrics)

               Model           MAE         NSE         MASE       MAE_HIGH
0    LSTM-EncDec-DFF  27.73 ± 0.58  0.75 ± 0.0  0.53 ± 0.01   109.59 ± 7.7
1  LSTM-EncDec-DFF-2  26.29 ± 0.85  0.76 ± 0.0   0.5 ± 0.02  105.62 ± 6.52


In [8]:
# Predictions per station
n_stations = 6
error_metrics = []

models = [
    "LSTM-EncDec-DFF",
    "LSTM-EncDec-DFF-2"
]

for model_name, model in zip(models,predictions_per_model):
  mae_high_per_station = []
  mae_per_station = []
  nse_per_station = []
  mase_per_station = []
  extreme_lows_per_station = []
  extreme_highs_per_station = []
  model =  np.array(model)
  model = model.transpose(1,2,0).reshape(-1,6)

  for i,name in zip(range(n_stations),weather_data.columns[7:]):

      df = pd.DataFrame({'timestamp': range(0,len(actuals[:,i]+1)),'Flow': actuals[:,i]})

      threshold_high = df['Flow'].quantile(0.95)  

      extreme_highs = df[df['Flow'] >= threshold_high]

      df2 = pd.DataFrame({'timestamp': range(0,len(actuals[:,i]+1)),'Flow': model[:,i].flatten()})

      threshold_high2 = df['Flow'].quantile(0.95)  

      extreme_highs2 = df2[df['Flow'] >= threshold_high2]

      extreme_highs_per_station.append(extreme_highs)

      mae_high = round(mean_absolute_error(extreme_highs['Flow'], extreme_highs2['Flow']),2)
      mae_high_per_station.append(mae_high)
      mae = round(mean_absolute_error(np.array(df['Flow']), np.array(df2['Flow'])),2)
      mae_per_station.append(mae)

      nse = round(nash_sutcliffe_efficiency(np.array(df['Flow']), np.array(df2['Flow'])),2)
      nse_per_station.append(nse)
      mase = round(calculate_mase(np.array(df['Flow']), np.array(df2['Flow']),20),2)
      mase_per_station.append(mase)

  error_metrics_df = pd.DataFrame({
      'Model' : model_name,
      'Station': [f'{name}' for name in weather_data.columns[7:]],
      'MAE': mae_per_station,
      'NSE' : nse_per_station,
      'MASE': mase_per_station,
      'MAE_HIGH': mae_high_per_station

  })

  error_metrics.append(error_metrics_df)

error_metrics = pd.concat(error_metrics,axis=0).reset_index(drop=True)
error_metrics.to_csv("error-metrics-un.csv")
print(error_metrics)

                Model Station    MAE   NSE  MASE  MAE_HIGH
0     LSTM-EncDec-DFF   54001  23.87  0.71  1.12     95.92
1     LSTM-EncDec-DFF   54005  18.29  0.66  1.13     67.03
2     LSTM-EncDec-DFF   54032  32.95  0.78  1.15     96.23
3     LSTM-EncDec-DFF   54057  39.58  0.77  1.15    130.81
4     LSTM-EncDec-DFF   54095  25.79  0.65  1.08    124.13
5     LSTM-EncDec-DFF   55002  25.88  0.48  1.03    142.14
6   LSTM-EncDec-DFF-2   54001  22.82  0.72  1.07     93.27
7   LSTM-EncDec-DFF-2   54005  17.28  0.68  1.07     63.85
8   LSTM-EncDec-DFF-2   54032  31.50  0.79  1.10     96.86
9   LSTM-EncDec-DFF-2   54057  37.59  0.78  1.09    121.66
10  LSTM-EncDec-DFF-2   54095  23.65  0.67  0.99    120.78
11  LSTM-EncDec-DFF-2   55002  24.92  0.50  0.99    137.73


In [9]:
models_df = pd.DataFrame({
    "Actual": actuals.ravel(),  
    "LSTM-EncDec-DFF": np.array(predictions_per_model[0]).transpose(1,0,2).ravel(),
    "LSTM-EncDec-DFF-2": np.array(predictions_per_model[1]).transpose(1,0,2).ravel(),
})

actual_values = models_df.pop("Actual")

errors_df = models_df.copy()
for model_name in models_df.columns:
    errors_df[model_name] = actual_values - models_df[model_name]
errors_melted = errors_df.melt(var_name="Model", value_name="Error")
model_names = models_df.columns

errors_melted.to_csv('errors_melted_un.csv')

sns.set_theme(style="white")

palette = sns.color_palette("tab10", n_colors=len(model_names))
model_colors = dict(zip(model_names, palette))  

cumulative_residuals = errors_df.cumsum()/errors_df.std()
window_size = max(5, len(errors_df) // 20)  
rolling_mean = cumulative_residuals.rolling(window=window_size, center=True).mean()

rolling_mean.to_csv('rolling_mean_un.csv')